# 🎚️ 26 · Every particle its own shape: the deep Gaussian mixture model (e2gmm)
### A neural network reads each particle and moves the Gaussians — motion and composition from one data set

**BiologyToResearch** · Part F — Conformational heterogeneity: Gaussian mixtures and neural fields · Module 26 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/26_e2gmm_heterogeneity.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 3–4 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn, PyTorch |
| **Prerequisites** | [25 · Molecules as clouds of Gaussians](25_gaussian_mixture_molecules.ipynb) (GMM, projection, FRC, Fourier reconstruction); [05 · Deep learning](05_deep_learning_pytorch.ipynb); [04 · Statistics](04_noise_statistics_inference.ipynb) (PCA, K-means) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Chen & Ludtke 2021 (deep mixed-dimensional GMM, *Nat Methods*); Chen, Toader & Lederman 2023 (scalable high-resolution deep GMM, *J Mol Biol*) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments are small, controlled *toy versions* of published methods (1,500 particles of 40 × 40 pixels, 128
> Gaussians, networks trained for seconds). They illustrate the core ideas; they do **not** reproduce the papers'
> architectures or numbers. **All molecules and images are generated by code** (nothing is downloaded); our particles are
> less noisy than real ones (SNR 0.3) and have no CTF, so that a CPU can learn from only 1,500 of them.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Simulate** a data set with two kinds of heterogeneity — a continuous **motion** (a swinging arm) and a discrete
   change of **composition** (a partner that is bound or not) — and **show** how a single consensus map blurs both.
2. **Explain** and **compute** the input that e2gmm's encoder reads: the gradient of the FRC with respect to the
   Gaussians of a neutral model, and **explain** why it does not depend on the particle's orientation.
3. **Build and train** an encoder–decoder that maps each particle to a point in a small **latent space** and back to a
   particle-specific Gaussian mixture, with the FRC as loss.
4. **Read** a latent space: principal components, decoded "eigen-trajectories", and reconstructions of particle subsets.
5. **Focus** the analysis on composition (amplitude-only) or on a domain's rigid-body motion (6-number decoder), and
   **explain** the coarse-to-fine transition matrix of Chen, Toader & Lederman (2023).
6. **Name** the main pitfalls: pose errors that masquerade as motion, empty regions of latent space, and the need to
   validate structures decoded from a network.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | tools, PyTorch helpers from Module 25 | 5 min |
| 1 · A moving, changing molecule | simulate 1,500 heterogeneous particles | 10 min |
| 2 · What one map hides | the consensus reconstruction | 5 min |
| 3 · The neutral Gaussian model | K-means + a short FRC refinement | 10 min |
| 4 · A pose-free fingerprint of each particle | FRC gradients; are they informative? | 15 min |
| 5 · Encoder and decoder | train e2gmm-lite; the latent space | 20 min |
| 6 · Reading the latent space | eigen-trajectory, subset reconstructions | 15 min |
| 7 · Focusing: composition and rigid bodies | amplitude-only, rigid-body arm, coarse-to-fine | 20 min |
| 8 · Pitfalls | what can fool the method | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Imagine a kinase whose regulatory arm swings open and shut, and whose activating partner binds only some of the time.
A cryo-EM sample of it is a mixture: every particle has its own arm angle, and some carry the partner. Averaging
everything gives one map in which the arm is a smear and the partner is a faint ghost. Classical **3D classification**
(Module 14) sorts particles into a few discrete boxes — fine for "partner yes / no", hopeless for a continuous swing.
Chen & Ludtke (2021) asked instead: *can a network learn, for every single particle, how the Gaussians of a model must
move or dim to match it?* The answers, collected over hundreds of thousands of particles, form a **map of the
conformational landscape** — the subject of this module and the next three.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from scipy import ndimage
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import cross_val_score

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
T_START = time.time()
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

Helpers copied from [`course_toolkit.py`](../course_toolkit.py): the test protein and `move_domain` (which turns one
domain about a pivot), the Gaussian-mixture helpers and `reconstruct_fourier` of Module 25, rotations (Module 02) and
`fsc` (Module 10).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def flexible_protein(seed: int = 7):
    """A code-generated ~80 kDa protein for heterogeneity experiments (Modules 25–30).

    Domains 0 and 1 form a rigid body; domain 2 (the 'arm') sits on domain 1 and can swing about a hinge; domain 3 is
    a small partner subunit that may be bound or not. Returns (atoms, hinge (x, y, z) in Å); atoms['domain'] labels
    every atom. Coordinates are in Å, centred near the origin.
    """
    rng = np.random.default_rng(seed)
    specs = [((21.0, 17.5, 16.0), (-19.0, -7.5, 0.0)), ((19.0, 16.0, 17.5), (19.0, -7.5, 0.0)),
             ((17.5, 12.5, 14.0), (21.0, 33.0, 0.0)), ((12.5, 11.0, 11.0), (-21.0, 26.0, 2.5))]
    parts = []
    for d, (radii, centre) in enumerate(specs):
        part = _transform(helix_bundle(radii, rng, chain="ABCD"[d]), _rz(rng.uniform(0, 2 * np.pi)), centre)
        part["domain"] = np.full(len(part["xyz"]), d)
        parts.append(part)
    atoms = merge_atoms(parts)
    shift = atoms["xyz"][atoms["domain"] < 3].mean(0)
    atoms["xyz"] = (atoms["xyz"] - shift).astype(np.float32)
    return atoms, np.array([19.0, 14.0, 0.0]) - shift


def move_domain(atoms, domain: int, R: np.ndarray, pivot) -> dict:
    """Copy of `atoms` with every atom of `domain` rotated by R (acting on (x, y, z)) about `pivot` (Å)."""
    out = dict(atoms)
    xyz = atoms["xyz"].astype(np.float64).copy()
    sel = atoms["domain"] == domain
    xyz[sel] = (xyz[sel] - pivot) @ np.asarray(R, float).T + pivot
    out["xyz"] = xyz.astype(np.float32)
    return out


def pseudoatoms_kmeans(vol: np.ndarray, n: int, voxel: float = 1.0, rng=None, iters: int = 30,
                       level: float = 0.05):
    """Summarise a density map by n Gaussian 'pseudo-atoms': weighted K-means on the voxels above level × max.

    Returns centres (n, 3) as (x, y, z) in Å relative to the box centre, and amplitudes (density in each cluster).
    """
    rng = np.random.default_rng(0) if rng is None else rng
    idx = np.argwhere(vol > level * vol.max())                                 # (z, y, x) of dense voxels
    w = vol[tuple(idx.T)].astype(np.float64)
    pts = ((idx[:, ::-1] - (np.array(vol.shape[::-1]) - 1) / 2.0) * voxel).astype(np.float32)
    c = pts[rng.choice(len(pts), n, replace=False, p=w / w.sum())].copy()
    for _ in range(iters):
        d2 = (pts ** 2).sum(1)[:, None] - 2 * pts @ c.T + (c ** 2).sum(1)[None]
        lab = d2.argmin(1)
        ws = np.bincount(lab, w, n)
        for a in range(3):
            upd = np.bincount(lab, w * pts[:, a], n)
            c[ws > 0, a] = upd[ws > 0] / ws[ws > 0]
    return c.astype(np.float32), np.bincount(lab, w, n).astype(np.float32)


def gmm_to_map(centres, amps, sigmas, box: int, voxel: float = 1.0) -> np.ndarray:
    """Density [z, y, x] (box³, origin at the box centre) of Σ_j A_j exp(−|x − c_j|² / 2σ_j²); c and σ in Å."""
    centres = np.asarray(centres, float)
    s = np.broadcast_to(np.asarray(sigmas, float), (len(centres),))
    g = (np.arange(box) - (box - 1) / 2.0) * voxel
    ex = [np.exp(-(g[None, :] - centres[:, a, None]) ** 2 / (2 * s[:, None] ** 2)) for a in range(3)]
    return np.einsum("n,nz,ny,nx->zyx", np.asarray(amps, float), ex[2], ex[1], ex[0], optimize=True).astype(np.float32)


def gmm_project(centres, amps, sigmas, R, box: int, pixel: float = 1.0) -> np.ndarray:
    """Projection [y, x] along z of a Gaussian mixture rotated by R (acting on (x, y, z)); same units as `gmm_to_map`.

    A 3D Gaussian projects to a 2D Gaussian of the same width, so the image is Σ_j w_j · g_y,j ⊗ g_x,j — two small
    matrices instead of a 3D grid (the 'project x and y separately' trick).
    """
    p = (np.asarray(centres, np.float32) @ np.asarray(R, np.float32).T)
    s = np.broadcast_to(np.asarray(sigmas, np.float32), (len(p),))[:, None]
    g = ((np.arange(box) - (box - 1) / 2.0) * pixel).astype(np.float32)[None, :]
    gx = np.exp(-(g - p[:, 0, None]) ** 2 / (2 * s ** 2))
    gy = np.exp(-(g - p[:, 1, None]) ** 2 / (2 * s ** 2))
    w = np.asarray(amps, np.float32)[:, None] * np.float32(np.sqrt(2 * np.pi)) * s / np.float32(pixel)   # ∫ dz
    return ((gy * w).T @ gx).astype(np.float32)


def reconstruct_fourier(images: np.ndarray, rotations, wiener: float = 1.0, chunk: int = 64) -> np.ndarray:
    """3D map [z, y, x] from projection images [n, y, x] with known rotations (R acts on (x, y, z), as in
    `rotate_volume`; image = rotate_volume(vol, R).sum(axis=0)).

    Projection-slice theorem: each image's 2D Fourier transform is the central slice of the 3D transform at
    k₃ = Rᵀ(kx, ky, 0). Slices are inserted with trilinear weights into a 2× padded grid, divided by the summed
    weights (+ `wiener`, in units of one image), transformed back and corrected for the interpolation kernel.
    """
    images = np.asarray(images, np.float64)
    n, N, _ = images.shape
    M, pad = 2 * N, N // 2
    k = np.arange(M) - M // 2
    ky, kx = np.meshgrid(k, k, indexing="ij")
    keep = (kx ** 2 + ky ** 2).ravel() <= (M // 2 - 1) ** 2
    plane = np.stack([kx.ravel(), ky.ravel(), np.zeros(M * M)], 1)[keep]
    shift_phase = np.exp(-1j * np.pi * (kx + ky) / M).ravel()[keep]          # half-pixel: image centre → index M/2
    Fr, Fi, W = np.zeros(M ** 3), np.zeros(M ** 3), np.zeros(M ** 3)
    corners = [(dz, dy, dx) for dz in (0, 1) for dy in (0, 1) for dx in (0, 1)]
    for start in range(0, n, chunk):
        idx_all, val_all, w_all = [], [], []
        for img, R in zip(images[start:start + chunk], list(rotations)[start:start + chunk]):
            big = np.zeros((M, M)); big[pad:pad + N, pad:pad + N] = img
            Fimg = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(big))).ravel()[keep] * shift_phase
            q = plane @ np.asarray(R, float)                                    # rows: (Rᵀ k)ᵀ in (x, y, z)
            pz = q[:, ::-1] + M // 2                                            # (z, y, x) grid coordinates
            i0 = np.floor(pz).astype(int); f = pz - i0
            for dz, dy, dx in corners:
                ii = i0 + (dz, dy, dx)
                wt = (np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1])
                      * np.where(dx, f[:, 2], 1 - f[:, 2]))
                ok = np.all((ii >= 0) & (ii < M), 1)
                idx_all.append((ii[ok, 0] * M + ii[ok, 1]) * M + ii[ok, 2]); val_all.append(wt[ok] * Fimg[ok])
                w_all.append(wt[ok])
        idx, val, wts = np.concatenate(idx_all), np.concatenate(val_all), np.concatenate(w_all)
        Fr += np.bincount(idx, val.real, M ** 3); Fi += np.bincount(idx, val.imag, M ** 3); W += np.bincount(idx, wts, M ** 3)
    F3 = ((Fr + 1j * Fi) / (W + wiener)).reshape(M, M, M)
    kz3, ky3, kx3 = np.meshgrid(k, k, k, indexing="ij")
    F3 *= np.exp(1j * np.pi * (kx3 + ky3 + kz3) / M)                            # centre back to (N − 1)/2
    vol = np.real(np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(F3))))[pad:pad + N, pad:pad + N, pad:pad + N]
    x = (np.arange(N) - (N - 1) / 2.0) / M
    sinc2 = np.sinc(x) ** 2                                                     # trilinear-kernel correction
    return (vol / (sinc2[:, None, None] * sinc2[None, :, None] * sinc2[None, None, :])).astype(np.float32)
print("🧰 toolkit loaded")

### PyTorch helpers (from Module 25)

The differentiable projector and FRC of Module 25, extended so that every particle in a batch may have **its own**
Gaussian centres and amplitudes (arrays of shape `(batch, n, 3)` and `(batch, n)`).

In [ ]:
# ── PyTorch GMM projector and FRC (Module 25, per-particle version) ──
N, PX = 40, 4.0                                                        # 40 × 40 pixels of 4 Å
GRID_T = torch.tensor((np.arange(N) - (N - 1) / 2) * PX, dtype=torch.float32)
_fy, _fx = np.fft.fftfreq(N), np.fft.rfftfreq(N)
RING_T = torch.tensor(np.minimum((np.sqrt(_fy[:, None] ** 2 + _fx[None, :] ** 2) * N).astype(int), N // 2).ravel())


def project_gmm_torch(centres, amps, sigma: float, R):
    """Projections (b, N, N) at poses R (b, 3, 3) of GMMs with centres (n, 3) or (b, n, 3) Å and amps (n,) or (b, n)."""
    p = torch.einsum("bij,nj->bni", R, centres) if centres.dim() == 2 else torch.einsum("bij,bnj->bni", R, centres)
    amps = amps[None].expand(R.shape[0], -1) if amps.dim() == 1 else amps
    gx = torch.exp(-(GRID_T - p[..., 0:1]) ** 2 / (2 * sigma ** 2))
    gy = torch.exp(-(GRID_T - p[..., 1:2]) ** 2 / (2 * sigma ** 2))
    return torch.einsum("bny,bnx->byx", gy * (amps * math.sqrt(2 * math.pi) * sigma / PX)[..., None], gx)


def frc_torch(a, b, rmin: int, rmax: int):
    """Per-image mean Fourier ring correlation over rings rmin..rmax between batches of images (b, N, N)."""
    Fa = torch.fft.rfft2(a).reshape(a.shape[0], -1)
    Fb = torch.fft.rfft2(b).reshape(b.shape[0], -1)
    ring_sum = lambda v: torch.zeros(a.shape[0], N // 2 + 1).index_add(1, RING_T, v)
    num = ring_sum((Fa * Fb.conj()).real)
    den = torch.sqrt(ring_sum(Fa.abs() ** 2) * ring_sum(Fb.abs() ** 2) + 1e-12)
    return (num / den)[:, rmin:rmax + 1].mean(1)


print("projector and FRC ready")

---
## 1 · A moving, changing molecule

> 📍 **Where we are.** Module 25 fitted *one* Gaussian mixture to all particles. Today every particle gets its own mixture.
> Module 27 will do the same with a different representation (a neural field), Module 28 will move it into cells
> (cryo-ET), and Module 29 will use per-particle motions to sharpen maps.

We take the protein of Module 25 and give it two kinds of variability:

- **Motion:** the arm (domain 2) swings about the hinge, about the z axis, by an angle $\theta$ drawn uniformly
  between −30° and +30° — a continuous **conformational landscape**.
- **Composition:** the partner (domain 3) is bound in 60 % of particles and absent in the rest.

We keep the true $\theta$ and the partner flag of every particle — **hidden from the method**, used only to grade it.

In [ ]:
# ── 1.1 Three conformations of the protein ──
atoms, hinge = flexible_protein()
Z_ATOM = np.array([{"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0}.get(e, 6.0) for e in atoms["element"]], np.float32)
SIG_ATOM = 2.5


def conformation(theta_deg: float, partner: bool):
    """Atom coordinates and amplitudes of the protein with the arm turned by theta (about z, at the hinge)."""
    moved = move_domain(atoms, 2, rot_z(np.radians(theta_deg)), hinge)
    amps = Z_ATOM * ((atoms["domain"] != 3) | partner)                   # partner atoms get zero amplitude if absent
    return moved["xyz"], amps.astype(np.float32)


fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.8))
for ax, (th, pa) in zip(axes, [(-30, True), (0, True), (30, True), (0, False)]):
    xyz, amp = conformation(th, pa)
    ax.imshow(gmm_project(xyz, amp, SIG_ATOM, np.eye(3), N, PX))
    ax.set_title(f"θ = {th:+d}°, partner {'bound' if pa else 'absent'}", fontsize=9); ax.axis("off")
plt.suptitle("The same protein in four states (seen down z)", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.2 Simulate 1,500 heterogeneous particles ──
N_PART, SNR = 1500, 0.3
theta_true = rng.uniform(-30, 30, N_PART)
partner_true = rng.random(N_PART) < 0.6
rot_true = np.array([random_rotation(rng) for _ in range(N_PART)]).astype(np.float32)
t0 = time.time()
clean = np.empty((N_PART, N, N), np.float32)
for i in range(N_PART):
    xyz, amp = conformation(theta_true[i], partner_true[i])
    clean[i] = gmm_project(xyz, amp, SIG_ATOM, rot_true[i], N, PX)
particles = (clean + rng.normal(0, np.sqrt(clean.var() / SNR), clean.shape)).astype(np.float32)
print(f"{N_PART} particles in {time.time() - t0:.1f} s; partner bound in {partner_true.mean():.0%}; "
      f"θ from {theta_true.min():.0f}° to {theta_true.max():.0f}°; SNR {SNR}")

---
## 2 · What one map hides

The standard first step is a **consensus** reconstruction: all particles, their (known) poses, one map (Module 25,
Section 2).

In [ ]:
# ── 2.1 The consensus map vs two of the true states ──
consensus = reconstruct_fourier(particles, rot_true)
truth_open = gmm_to_map(*conformation(-30, True), SIG_ATOM, N, PX)
truth_shut = gmm_to_map(*conformation(30, True), SIG_ATOM, N, PX)
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
for ax, v, t in zip(axes, [truth_open, truth_shut, consensus], ["truth, θ = −30°", "truth, θ = +30°", "consensus (all 1,500)"]):
    ax.imshow(lowpass(v, PX / 10).sum(axis=0)); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the consensus the body is crisp, but the arm is a fan-shaped smear covering all its
positions, and the partner is weaker than the body (it is present in only 60 % of the particles). Everything that
varies is averaged into a blur — Module 10 (Section 6.4) showed the same effect as poorer *local resolution*.

---
## 3 · The neutral Gaussian model

e2gmm needs a **neutral model**: a GMM that represents the average molecule, from which each particle's mixture will
deviate. We make one as in Module 25 — K-means pseudo-atoms of the (lightly low-passed) consensus map, then a short FRC
refinement against the particles, with the FRC limited to 20 Å (motions are visible at low resolution, and the papers
also restrict the resolution while learning heterogeneity).

In [ ]:
# ── 3.1 Neutral GMM: K-means of the consensus + FRC refinement ──
N_G, SIG_G, RMAX = 128, 4.0, int(N * PX / 20.0)                 # 128 Gaussians of width 4 Å; FRC up to 20 Å
cen0, amp0 = pseudoatoms_kmeans(np.clip(lowpass(consensus, PX / 12), 0, None), N_G, voxel=PX,
                                rng=np.random.default_rng(1))
X_t, R_t = torch.tensor(particles), torch.tensor(rot_true)
cen_t = torch.tensor(cen0, requires_grad=True)
loga_t = torch.tensor(np.log(amp0 / amp0.max()), requires_grad=True)
opt = torch.optim.Adam([{"params": [cen_t], "lr": 0.3}, {"params": [loga_t], "lr": 0.02}])
t0 = time.time()
for it in range(150):
    idx = torch.randint(0, N_PART, (48,))
    loss = -frc_torch(project_gmm_torch(cen_t, loga_t.exp(), SIG_G, R_t[idx]), X_t[idx], 1, RMAX).mean()
    opt.zero_grad(); loss.backward(); opt.step()
neutral_c, neutral_a = cen_t.detach(), loga_t.exp().detach()
print(f"neutral model: {N_G} Gaussians; mean FRC to 20 Å = {-loss.item():.3f}  [{time.time() - t0:.1f} s]")

---
## 4 · A pose-free fingerprint of each particle

The encoder must read a particle and say how it differs from the neutral model. Feeding it the raw image would force
the network to also learn about orientations — the same molecule looks completely different from different
directions. Chen & Ludtke's trick: describe each particle by **how the neutral model would have to change to fit it
better**,

$$\mathbf g_i \;=\; \frac{\partial\,\mathrm{FRC}\big(\mathrm{Proj}(\text{GMM}_{\text{neutral}}, R_i),\,X_i\big)}
  {\partial\,(\mathbf c_1, A_1, \dots, \mathbf c_n, A_n)},$$

*in words:* for particle $i$, project the neutral model at the particle's pose, compare with the particle (FRC), and ask
autograd in which direction each Gaussian's centre and amplitude should move to raise the score. Because the
derivative is taken with respect to the Gaussians **in the molecule's own frame**, a particle seen from the top and
one seen from the side that have the same arm angle produce similar fingerprints: the pose has been factored out. (The
paper uses all five parameters per Gaussian; we keep the width fixed, so 4 × 128 = 512 numbers per particle.)

💡 **Per-particle gradients in one go.** Autograd sums gradients over a batch. To get one gradient *per particle* we
give every particle in the batch its own copy of the neutral parameters; each copy only influences its own particle.

In [ ]:
# ── 4.1 FRC-gradient fingerprints of all particles ──
t0 = time.time()
neutral_params = torch.cat([neutral_c, neutral_a[:, None]], 1)                     # (n, 4): x, y, z, amplitude
feats = []
for s in range(0, N_PART, 100):
    b = min(100, N_PART - s)
    copies = neutral_params[None].expand(b, -1, -1).clone().requires_grad_(True)  # one copy per particle
    score = frc_torch(project_gmm_torch(copies[..., :3], copies[..., 3], SIG_G, R_t[s:s + b]), X_t[s:s + b], 1, RMAX)
    grad, = torch.autograd.grad(score.sum(), copies)
    feats.append(grad.reshape(b, -1))
feats = torch.cat(feats)
feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)                             # standardise each feature
print(f"fingerprints: {tuple(feats.shape)} in {time.time() - t0:.1f} s")

Before building any network, a quick **sanity check**: do the fingerprints carry the information we hope for? We use
the hidden truth *only for grading*: a linear regression from fingerprint to arm angle, and a logistic regression from
fingerprint to partner flag, both evaluated by 3-fold cross-validation (Module 05: never grade on the training data).

In [ ]:
# ── 4.2 Are the fingerprints informative? (linear probes, cross-validated) ──
F_np = feats.numpy()
r2_theta = cross_val_score(RidgeCV(alphas=[1, 10, 100, 1000]), F_np, theta_true, cv=3).mean()
acc_partner = cross_val_score(LogisticRegression(C=0.1, max_iter=3000), F_np, partner_true, cv=3).mean()
print(f"arm angle from fingerprint: R² = {r2_theta:.2f} (0 = no information, 1 = perfect)")
print(f"partner from fingerprint:   accuracy {acc_partner:.2f} (always guessing 'bound' gives {partner_true.mean():.2f})")
assert r2_theta > 0.3 and acc_partner > partner_true.mean() + 0.1, "fingerprints should carry the hidden variables"

🔍 **What just happened?** Even a straight-line model recovers a good part of the arm angle and most partner flags from
the 512-number fingerprints of single, noisy particles in random orientations. The information is there; the network's
job is to organise it **without** ever being told $\theta$ or the partner flag.

---
## 5 · Encoder and decoder

The architecture of e2gmm, in miniature:

| part | input → output | our size (paper's) |
|---|---|---|
| **encoder** | fingerprint $\mathbf g_i$ (512) → latent point $\mathbf z_i$ | 2 hidden layers of 256 (3 × 512), latent 4-D (default 4-D) |
| **decoder** | $\mathbf z$ → change of every Gaussian (Δcentre, Δamplitude) | 2 hidden layers of 256 (3 × 512) |
| **loss** | FRC between the decoded GMM projected at pose $R_i$ and particle $X_i$ | same (FRC) |

Three details matter:

- **Start at neutral.** The decoder's last layer starts at zero, so at first *every* latent point decodes to the neutral
  model. (The paper pre-trains its decoder so that $\mathbf z = 0$ gives the neutral model; this has the same effect.)
- **A little noise in the latent space.** During training a small random vector is added to $\mathbf z$ before decoding
  (fixed, not learned — "VAE-like" in the authors' words), so nearby latent points must decode to similar molecules.
- **Keep latent points near the origin** with a small penalty on $\lVert\mathbf z\rVert^2$ (as in the e2gmm code).

In [ ]:
# ── 5.1 e2gmm-lite: networks and training loop ──
def mlp(n_in: int, n_out: int, hidden: int = 256) -> nn.Sequential:
    """Two hidden ReLU layers."""
    return nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_out))


def decode(decoder, z, mode: str, arm_mask=None):
    """Particle-specific GMMs (centres (b, n, 3), amplitudes (b, n)) from latent points z, for one of four modes."""
    out, b = decoder(z), z.shape[0]
    if mode == "free":                                    # every centre and amplitude may change
        d = out.reshape(b, N_G, 4)
        return neutral_c[None] + 10.0 * d[..., :3], neutral_a[None] * torch.exp(d[..., 3])
    if mode == "amplitude":                               # only amplitudes change (composition)
        return neutral_c[None].expand(b, -1, -1), neutral_a[None] * torch.exp(out)
    if mode == "rigid":                                   # one rigid motion (rotation + shift) of the arm's Gaussians
        R = axis_angle_matrix_torch(0.5 * out[:, :3])
        moved = torch.einsum("bij,nj->bni", R, neutral_c - HINGE_T) + HINGE_T + 5.0 * out[:, None, 3:]
        return torch.where(arm_mask[None, :, None], moved, neutral_c[None]), neutral_a[None].expand(b, -1)
    raise ValueError(mode)


def train_e2gmm(mode: str, z_dim: int = 4, epochs: int = 15, arm_mask=None, seed: int = SEED):
    """Train encoder + decoder with the FRC loss; returns (encoder, decoder, latent points of all particles)."""
    torch.manual_seed(seed)
    n_out = {"free": 4 * N_G, "amplitude": N_G, "rigid": 6}[mode]
    encoder, decoder = mlp(4 * N_G, z_dim), mlp(z_dim, n_out)
    nn.init.zeros_(decoder[-1].weight); nn.init.zeros_(decoder[-1].bias)     # start: every z decodes to neutral
    opt = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
    t0 = time.time()
    for ep in range(epochs):
        for idx in torch.randperm(N_PART).split(64):
            z = encoder(feats[idx])
            c, a = decode(decoder, z + 0.05 * torch.randn_like(z), mode, arm_mask)
            loss = -frc_torch(project_gmm_torch(c, a, SIG_G, R_t[idx]), X_t[idx], 1, RMAX).mean() \
                   + 1e-3 * (z ** 2).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        latent = encoder(feats).numpy()
    print(f"[{mode}] {epochs} epochs in {time.time() - t0:.0f} s; final training FRC {-loss.item():.3f}")
    return encoder, decoder, latent


def axis_angle_matrix_torch(v):
    """Rotation matrices (b, 3, 3) from axis-angle vectors v (b, 3) (direction = axis, length = angle in rad)."""
    angle = v.norm(dim=1, keepdim=True).clamp_min(1e-8)
    k = v / angle
    K = torch.zeros(v.shape[0], 3, 3)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 2] = -k[:, 2], k[:, 1], -k[:, 0]
    K[:, 1, 0], K[:, 2, 0], K[:, 2, 1] = k[:, 2], -k[:, 1], k[:, 0]
    s, c = torch.sin(angle)[..., None], torch.cos(angle)[..., None]
    return torch.eye(3)[None] + s * K + (1 - c) * K @ K


HINGE_T = torch.tensor(hinge, dtype=torch.float32)
print("networks defined")

In [ ]:
# ── 5.2 Train the full ("free") model ──
enc_free, dec_free, z_free = train_e2gmm("free")

In [ ]:
# ── 5.3 The latent space, coloured by the hidden truth ──
pca_free = PCA(2).fit(z_free)
pc_free = pca_free.transform(z_free)
rho_pc = [spearmanr(pc_free[:, k], theta_true).correlation for k in range(2)]
acc_free = LogisticRegression().fit(z_free, partner_true).score(z_free, partner_true)
print(f"Spearman correlation with the arm angle: PC1 {rho_pc[0]:+.2f}, PC2 {rho_pc[1]:+.2f}")
print(f"partner flag readable from the 4-D latent (logistic regression): {acc_free:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
sc = axes[0].scatter(pc_free[:, 0], pc_free[:, 1], c=theta_true, cmap="coolwarm", s=5)
plt.colorbar(sc, ax=axes[0], label="true arm angle θ (°)")
axes[1].scatter(pc_free[~partner_true, 0], pc_free[~partner_true, 1], s=5, color="tab:gray", label="partner absent")
axes[1].scatter(pc_free[partner_true, 0], pc_free[partner_true, 1], s=5, color="tab:green", alpha=0.5, label="partner bound")
axes[1].legend(fontsize=7, markerscale=3)
for ax in axes:
    ax.set_xlabel("latent PC1"); ax.set_ylabel("latent PC2")
axes[0].set_title("coloured by arm angle (hidden)", fontsize=9); axes[1].set_title("coloured by partner (hidden)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Nobody told the network about arms or partners, yet its latent space is organised by
them: one principal direction runs along the arm angle (the colour gradient in the left panel), and the partner flag is
largely readable from the latent point. With real data you would not have the colours — you would *discover* such
structure by looking at the latent space and decoding it (next section). Our toy is far from perfect (single noisy
40-pixel images are very little evidence); the paper works with $10^5$ particles of 100–300 pixels.

---
## 6 · Reading the latent space

### 6.1 Decoding an "eigen-trajectory"

Chen & Ludtke analyse the latent space with **PCA** and decode points along each principal axis back into Gaussian
mixtures: each axis becomes a movie of a motion (an "eigen-trajectory"). Let us walk along PC1 from −2 to +2 standard
deviations.

In [ ]:
# ── 6.1 Decode five points along latent PC1 ──
pc1_dir, pc1_sd = pca_free.components_[0], pc_free[:, 0].std()
steps = [-2, -1, 0, 1, 2]
z_path = torch.tensor(np.array([pca_free.mean_ + s * pc1_sd * pc1_dir for s in steps]), dtype=torch.float32)
with torch.no_grad():
    c_path, a_path = decode(dec_free, z_path, "free")
neutral_map = gmm_to_map(neutral_c.numpy(), neutral_a.numpy(), SIG_G, N, PX)
fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for j, s in enumerate(steps):
    m = gmm_to_map(c_path[j].numpy(), a_path[j].numpy(), SIG_G, N, PX)
    axes[0, j].imshow(m.sum(axis=0)); axes[0, j].set_title(f"PC1 = {s:+d} sd", fontsize=9)
    axes[1, j].imshow((m - neutral_map).sum(axis=0), cmap="RdBu_r", vmin=-0.6 * neutral_map.sum(0).max() / 4,
                      vmax=0.6 * neutral_map.sum(0).max() / 4)
for ax in axes.ravel():
    ax.axis("off")
axes[1, 0].set_title("difference from neutral", fontsize=8, loc="left")
plt.suptitle("An eigen-trajectory: GMMs decoded along latent PC1 (projections down z)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 The decoded molecules differ mainly in the arm: in the difference maps (red = more density than neutral, blue = less)
density moves from one side of the arm's fan to the other as we walk along PC1 — the swing, recovered as a motion of
Gaussians. In real studies this is how motions are first seen: the authors report, for example, a tilt of about 8° of
the central protuberance in assembling ribosomal large subunits and a previously unreported motion in a spliceosome.

### 6.2 Back to the data: reconstruct particle subsets

Decoded GMMs are the network's *interpretation*. A safer step, used by all heterogeneity papers, is to go back to the
particles: split them by their latent coordinate and reconstruct each subset **directly from the images** (Module 25),
with no network in the loop.

In [ ]:
# ── 6.2 Three subsets along PC1 → three independent reconstructions ──
edges = np.quantile(pc_free[:, 0], [0, 1 / 3, 2 / 3, 1])
subset_maps, subset_theta = [], []
for k in range(3):
    sel = (pc_free[:, 0] >= edges[k]) & (pc_free[:, 0] <= edges[k + 1])
    subset_maps.append(reconstruct_fourier(particles[sel], rot_true[sel]))
    subset_theta.append(theta_true[sel].mean())
    print(f"subset {k + 1}: {sel.sum()} particles, mean true θ = {subset_theta[-1]:+.1f}° (hidden from the method)")
fig, axes = plt.subplots(1, 3, figsize=(8, 2.8))
for ax, m, th in zip(axes, subset_maps, subset_theta):
    ax.imshow(lowpass(m, PX / 10).sum(axis=0)); ax.set_title(f"PC1 tertile, mean θ {th:+.0f}°", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Particles sorted by the network's PC1 have clearly different average arm angles, and the
three maps reconstructed from the raw images show the arm in three positions — the network's suggestion, confirmed by
the data alone. (Each subset is still a mixture, so each arm is still somewhat blurred; finer bins need more particles.)

---
## 7 · Focusing the analysis: composition and rigid bodies

### 7.1 Amplitude-only: "which parts are there?"

A free decoder mixes motion and composition in one latent space. e2gmm lets the user choose **which parameters may
vary** (in the code, a mask over positions, amplitudes and widths). Let only the **amplitudes** change and the network
can only explain particles by Gaussians appearing and disappearing — composition. The authors used this mode on
ribosomal large subunits caught at different stages of assembly and report six clusters matching known assembly states.

In [ ]:
# ── 7.1 Amplitude-only model → two classes → two reconstructions ──
enc_amp, dec_amp, z_amp = train_e2gmm("amplitude")
km = KMeans(2, n_init=10, random_state=SEED).fit(z_amp)
cls = km.labels_
with torch.no_grad():
    _, a_cent = decode(dec_amp, torch.tensor(km.cluster_centers_, dtype=torch.float32), "amplitude")
partner_g = torch.tensor(np.linalg.norm(neutral_c.numpy() - atoms["xyz"][atoms["domain"] == 3].mean(0), axis=1) < 15)
bound_cls = int(np.argmax(a_cent[:, partner_g].sum(1).numpy()))          # the class whose partner Gaussians are brighter
agree = np.mean((cls == bound_cls) == partner_true)
print(f"class with brighter partner Gaussians: {bound_cls}; agreement of the 2 classes with the hidden partner flag: {agree:.2f}")
fig, axes = plt.subplots(1, 2, figsize=(5.6, 2.8))
for ax, k, t in zip(axes, [bound_cls, 1 - bound_cls], ["class 'partner bound'", "class 'partner absent'"]):
    m = reconstruct_fourier(particles[cls == k], rot_true[cls == k])
    ax.imshow(lowpass(m, PX / 10).sum(axis=0)); ax.set_title(f"{t}\n{(cls == k).sum()} particles", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With amplitudes as the only knob, the two latent clusters line up with "partner bound" and
"partner absent", and the two maps reconstructed from the raw particles show the partner at full strength in one and
much weaker in the other (the classes agree with the hidden truth for about nine particles in ten, so a little partner
density remains). Restricting what the decoder may change is a way of telling the method **what
kind of answer you are looking for**.

### 7.2 Rigid bodies: one domain, six numbers (Chen, Toader & Lederman 2023)

Many motions are domains moving as rigid units. Chen, Toader & Lederman (2023) added **rigid-body decoders**: for a
selected group of Gaussians, the decoder outputs only six numbers per particle — a rotation (3) and a translation (3).
Fewer outputs mean a much more constrained, more interpretable answer. We select the neutral Gaussians in the arm's
region and let a decoder rotate them about the hinge; the rotation angle about z can then be read off directly and
compared with the hidden truth.

In [ ]:
# ── 7.2 A rigid-body decoder for the arm ──
nc = neutral_c.numpy()
arm_mask = torch.tensor((nc[:, 1] > hinge[1] + 4) & (nc[:, 0] > 0))         # Gaussians above the hinge, arm side
print(f"{int(arm_mask.sum())} of {N_G} Gaussians assigned to the arm")
enc_rig, dec_rig, z_rig = train_e2gmm("rigid", arm_mask=arm_mask)
with torch.no_grad():
    R_pred = axis_angle_matrix_torch(0.5 * dec_rig(enc_rig(feats))[:, :3]).numpy()
angle_pred = np.degrees(np.arctan2(R_pred[:, 1, 0], R_pred[:, 0, 0]))       # rotation about z of each particle's arm
r_rig = np.corrcoef(angle_pred, theta_true)[0, 1]
slope, offset = np.polyfit(theta_true, angle_pred, 1)
print(f"predicted arm rotation vs hidden θ: r = {r_rig:.2f}; fit: predicted ≈ {slope:.2f} × θ {offset:+.1f}°")
fig, ax = plt.subplots(figsize=(4.4, 3.4))
ax.scatter(theta_true, angle_pred, s=4, alpha=0.4); ax.plot([-30, 30], [-30, 30], "k--", lw=1, label="perfect")
ax.set_xlabel("true arm angle θ (°)"); ax.set_ylabel("decoded rotation about z (°)"); ax.legend(fontsize=8)
ax.set_title(f"rigid-body decoder: r = {r_rig:.2f}", fontsize=9); plt.tight_layout(); plt.show()
assert r_rig > 0.5, "the rigid-body decoder should track the arm angle"

🔍 **What just happened?** With just six numbers per particle, the decoder recovers each particle's arm angle in physical
units (degrees): the points follow the diagonal. The decoded angles are somewhat compressed (slope below 1) — a
typical bias of noisy per-particle estimates, which shrink toward the average. This is the kind of readout the authors
report for real proteins, e.g. a helix rotation of about 6° in the TRPV1 ion channel and a tilt of about 30° of the
extracellular domain of the GLP-1 receptor.

### 7.3 Coarse Gaussians driving fine ones

With thousands of Gaussians (one per atom), letting the decoder move each independently is slow to train and can tear
the molecule apart. Chen, Toader & Lederman (2023) let a **small** GMM (tens of Gaussians, e.g. 32 from K-means) drive
the **large** one through a fixed **transition matrix** whose weights fall with distance, $T_{mj}=\exp(-50\,D_{mj}^2)$,
with $D$ the distance between coarse Gaussian $m$ and fine Gaussian $j$ (in units of the box size, as e2gmm stores
coordinates). The decoder outputs coarse displacements; each fine Gaussian moves with a weighted mix of its coarse
neighbours. Let us build $T$ for our model and watch one coarse Gaussian drag its neighbourhood.

In [ ]:
# ── 7.3 The coarse-to-fine transition matrix ──
coarse = KMeans(12, n_init=10, random_state=SEED).fit(nc).cluster_centers_                 # 12 coarse Gaussians (Å)
D = np.linalg.norm(coarse[:, None, :] - nc[None, :, :], axis=2) / (N * PX)                  # distances in box units
T = np.exp(-50 * D ** 2)                                                                     # (12 coarse × 128 fine) weights
arm_centre = atoms["xyz"][atoms["domain"] == 2].mean(0)
m_arm = int(np.argmin(np.linalg.norm(coarse - arm_centre, axis=1)))                          # the coarse Gaussian on the arm
dcoarse = np.zeros((12, 3)); dcoarse[m_arm] = [-12.0, 0, 0]                                 # move it 12 Å along −x
dfine = T.T @ dcoarse                                                                        # induced fine displacements
print(f"coarse Gaussian {m_arm} moved by 12 Å; fine Gaussians moving > 6 Å: {(np.linalg.norm(dfine, axis=1) > 6).sum()},"
      f" > 1 Å: {(np.linalg.norm(dfine, axis=1) > 1).sum()} of {N_G}")
fig, ax = plt.subplots(figsize=(4.8, 4))
ax.scatter(nc[:, 0], nc[:, 1], s=8, color="0.6", label="fine (neutral)")
ax.quiver(nc[:, 0], nc[:, 1], dfine[:, 0], dfine[:, 1], angles="xy", scale_units="xy", scale=1, color="tab:red", width=0.004)
ax.scatter(coarse[:, 0], coarse[:, 1], s=60, marker="s", facecolor="none", edgecolor="k", label="coarse")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.legend(fontsize=7, loc="lower left")
ax.set_title("one coarse move → a smooth fine displacement field", fontsize=9); plt.tight_layout(); plt.show()

🔍 Fine Gaussians near the moved coarse Gaussian follow it almost fully, those farther away less, and distant ones not
at all (each fine Gaussian moves by the weighted sum of its coarse neighbours' moves): the matrix turns a dozen numbers
into a smooth deformation of the whole molecule. (With $e^{-50D^2}$ the weight
falls to $1/e$ at $D\approx0.14$ box lengths — about 23 Å in our 160 Å box.) The 2023 paper combines this with an
atomic-model start, a small restraint on Cα–Cα distances and pseudo-bond angles, and a two-round fit (low resolution
first) — ingredients we will meet again in Module 30.

---
## 8 · Pitfalls

- **Pose errors look like motion.** The fingerprints assume each particle's pose is right. A particle whose pose is off
  by a few degrees will be "explained" by moving Gaussians — a fake motion. The authors list misalignment among the
  method's limitations and allow an optional orientation refinement against the particle-specific GMMs.
- **Empty latent regions.** Decoding a latent point where no particles live produces a molecule nobody observed.
  Always check the density of particles along a trajectory.
- **Decoded ≠ measured.** Treat decoded GMMs as hypotheses; confirm them by reconstructing particle subsets (Section
  6.2) and validating with half-maps (Module 10).

✏️ **Your turn.** Simulate pose errors: before cell 4.1, set
`R_t = torch.tensor(np.array([R @ rot_x(np.radians(rng.normal(0, 4))) @ rot_y(np.radians(rng.normal(0, 4)))
for R in rot_true]), dtype=torch.float32)` (poses wrong by a few degrees; `random_rotation` would not do — it is
uniform over all orientations), then re-run 4.1–5.3. Does the latent space still follow θ? Does it pick up anything new?

### 📄 Paper connection — Chen & Ludtke 2021: heterogeneity with a deep mixed-dimensional GMM

**Paper:** Chen M, Ludtke SJ. *Deep learning-based mixed-dimensional Gaussian mixture model for characterizing
variability in cryo-EM.* Nature Methods 18:930–936 (2021). [doi:10.1038/s41592-021-01220-5](https://doi.org/10.1038/s41592-021-01220-5)
(preprint [arXiv:2101.10356](https://arxiv.org/abs/2101.10356))

- **Question it answers:** how to characterise continuous motions and compositional changes in a cryo-EM data set
  without forcing particles into a few discrete classes.
- **What the paper actually does:** a neutral GMM is fitted to the consensus; the encoder reads, for each particle,
  the gradient of the FRC with respect to all Gaussian parameters of the neutral model (computed in the model's frame,
  so it does not depend on the particle's orientation; the code normalises it by its global spread); a decoder maps the
  latent point (default 4-D) to the parameters of a particle-specific GMM; encoder and decoder have three dense layers of
  512 units with ReLU, dropout and batch normalisation, and are trained with the FRC on particles of known orientation,
  after pre-training the decoder so that $\mathbf z = 0$ gives the neutral model; a small fixed noise is added to the
  latent. "Mixed-dimensional": the model is 3D, the data 2D, and the latent space low-dimensional. Focused analyses
  restrict which parameters (positions, amplitudes, widths) or which Gaussians may vary. Latent spaces are explored with
  UMAP/clustering and PCA eigen-trajectories.
- **Result reported:** the authors report, for example, six clusters matching known assembly states for ribosomal large
  subunits (EMPIAR-10076, amplitude-only), motions of the helicase and SF3b regions and a new mode in a spliceosome
  (EMPIAR-10180), and a tilt of about 11° of a helix in SARS-CoV-2 spike (EMPIAR-10492); on simulated data a linear
  motion gave a one-dimensional latent that tracked the truth.
- **What our toy keeps:** neutral GMM, FRC-gradient input, encoder/decoder with latent noise and length penalty, FRC loss,
  free and amplitude-only modes, PCA eigen-trajectory, subset reconstruction.
- **What our toy leaves out:** CTF, widths as variables, dropout/batch norm, UMAP, orientation refinement, GPU-scale data.
- **Limitations stated:** needs good orientations and a rigid-enough neutral map; misalignment can look like motion;
  GPU memory limits the number of Gaussians.
- **Code:** EMAN2 `e2gmm_refine.py`, GUI `e2gmm.py`, `e2gmm_analysis.py` (<https://github.com/cryoem/eman2>).

### 📄 Paper connection — Chen, Toader & Lederman 2023: scaling up, models and rigid bodies

**Paper:** Chen M, Toader B, Lederman R. *Integrating molecular models into CryoEM heterogeneity analysis using scalable
high-resolution deep Gaussian mixture models.* Journal of Molecular Biology 435(9):168014 (2023).
[doi:10.1016/j.jmb.2023.168014](https://doi.org/10.1016/j.jmb.2023.168014)

- **Question it answers:** the 2021 network converged slowly with thousands of Gaussians and ran out of memory; can the
  approach reach high resolution and use atomic models?
- **What the paper actually does:** initialises the GMM from an atomic model, from map pseudo-atoms, or both; projects
  Gaussians separably (Module 25); drives a large GMM with a small one through a transition matrix with weights
  $\exp(-50D^2)$; adds rigid-body decoders (six numbers per particle for a selected domain) and multi-decoder designs
  (one encoder, several decoders summed); fits in two rounds (low resolution first) with a small restraint on Cα–Cα
  distances and pseudo-bond angles.
- **Result reported:** for TRPV1 (EMPIAR-10059) a rotation of about 6° of a helix of the ankyrin-repeat domain; for the
  GLP-1 receptor (EMPIAR-10346, ≈ 2,100 Gaussians at 4 Å) a tilt of about 30° of the extracellular domain and shifts of
  transmembrane helix 1 of up to 8 Å.
- **What our toy keeps:** rigid-body decoder for one domain, the transition-matrix construction.
- **What our toy leaves out:** atomic-model initialisation and restraints (Module 30), multi-decoders, high resolution.

---
## ✅ Checkpoint

1. Why does e2gmm feed the encoder FRC gradients instead of the particle images?
   <details><summary>Answer</summary>The gradient is taken with respect to the Gaussians in the molecule's own frame, so it describes how the molecule differs from the neutral model independently of the particle's orientation; raw images mix shape and pose.</details>
2. What does the "free", the "amplitude-only" and the "rigid-body" decoder each allow to change?
   <details><summary>Answer</summary>Free: every centre and amplitude. Amplitude-only: only how much density each Gaussian has (composition). Rigid-body: one rotation and translation of a selected group of Gaussians (six numbers).</details>
3. Why start the decoder so that every latent point decodes to the neutral model?
   <details><summary>Answer</summary>Training then begins from a model that already fits the average; the network only has to learn deviations, which is easier and more stable.</details>
4. What is an eigen-trajectory?
   <details><summary>Answer</summary>The sequence of molecules decoded from latent points along a principal axis of the latent distribution — a movie of the main mode of variation.</details>
5. Why reconstruct particle subsets from the raw images after the network has done its work?
   <details><summary>Answer</summary>To confirm the network's interpretation with data alone: a decoded GMM is a model output and could be an artefact; a subset reconstruction is not.</details>
6. In $T_{mj}=e^{-50D^2}$, how far (in box lengths) must a fine Gaussian be from a coarse one for the weight to drop to 1/e?
   <details><summary>Answer</summary>50D² = 1 → D = 1/√50 ≈ 0.14 box lengths.</details>

## 🧠 Questions & discussion

1. Our arm moved about one axis only. What would you expect in the latent space if it also tilted out of plane,
   independently?
2. The partner was either fully bound or absent. How would a partially flexible partner (bound but wobbling) show up in
   the free and in the amplitude-only models?
3. Classical 3D classification (Module 14) would need how many classes to describe a continuous 60° swing at 10°
   precision? What is gained and lost with a continuous latent space?
4. The rigid-body decoder needed us to say *which* Gaussians form the arm. How might you find domains automatically from
   a free model's motions?
5. Why might the decoded angles be compressed toward zero (slope < 1)? Think of what the network should answer when a
   particle is too noisy to judge.

## 🚀 Bonus — going further

- **More particles, less noise.** Double `N_PART` or set `SNR = 1.0` and re-run: how do the probe R², the latent
  correlation and the rigid-body slope change?
- **A second motion.** Add an out-of-plane tilt of the arm (`rot_x`) with its own random angle; does a 2-D latent
  separate the two motions?
- **Hierarchical training.** Make the free decoder output 12 coarse displacements and apply `T.T @ d` (Section 7.3) to
  move the 128 Gaussians. Compare training speed and the smoothness of the decoded motions.
- **Real software:** follow the e2gmm tutorial in the EMAN2 documentation with a public data set from EMPIAR.

## 📝 Lab note

```text
Module 26 — date:
Input: 1,500 simulated particles (arm swing θ ∈ [−30°, 30°], partner bound in 60 %), SNR 0.3, known poses
Output: consensus map, neutral GMM (128 G), FRC-gradient fingerprints, latent spaces (free / amplitude / rigid)
Method: FRC gradient features, encoder–decoder with FRC loss, PCA eigen-trajectory, subset reconstruction, K-means
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Chen & Ludtke 2021; Chen, Toader & Lederman 2023
```

## 📚 References

1. Chen M, Ludtke SJ. Deep learning-based mixed-dimensional Gaussian mixture model for characterizing variability in cryo-EM. *Nat Methods* 18:930–936 (2021). doi:10.1038/s41592-021-01220-5
2. Chen M, Toader B, Lederman R. Integrating molecular models into CryoEM heterogeneity analysis using scalable high-resolution deep Gaussian mixture models. *J Mol Biol* 435(9):168014 (2023). doi:10.1016/j.jmb.2023.168014
3. Davis JH *et al.* Modular assembly of the bacterial large ribosomal subunit. *Cell* 167:1610–1622 (2016). doi:10.1016/j.cell.2016.11.020 — the assembly data set (EMPIAR-10076) used by many heterogeneity papers.
4. Sorzano COS *et al.* Survey of the analysis of continuous conformational variability of biological macromolecules by electron microscopy. *Acta Cryst F* 75:19–32 (2019). doi:10.1107/S2053230X18015108

## ➡️ Next

**[27 · Neural fields for heterogeneous structures: cryoDRGN](27_cryodrgn_neural_fields.ipynb)** — instead of Gaussians,
a network that *is* the map: give it a 3D frequency and a latent point, it returns the Fourier value.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).

In [ ]:
# ── Total run time ──
print(f"Notebook finished in {time.time() - T_START:.0f} s")